# 365 Probabilidades - Dia #159
## Qual a probabilidade de as oito horas de sono serem uma invenção moderna?

**Tipo:** Descritivo  
**Data de publicação:** 2026-11-19  
**Ferramenta:** Python  
**Decisão analisada:** Preciso mesmo dormir oito horas?  
**Hashtag:** #365Probabilidades #Dia159

---

### 📖 A História

[ABERTURA DA ANA - a sugestão que você recebeu, em uma linha sua]

Oito horas é o número que todo mundo sabe de cor. É também a conta que a gente faz ao acordar, para ver quanto faltou.

Fui ver de onde vem esse número, e se alguém já dormiu isso.

---

### 📚 O Conceito

A ideia comum é que a luz elétrica, as telas e a pressa encurtaram o nosso sono, e que antes se dormia mais.

Não dá para medir o sono da pré-história. O que dá para medir é o sono de povos que vivem hoje sem luz elétrica, caçando, coletando e plantando, e comparar com o sono medido em laboratório nas últimas décadas. Nenhum dos dois é a pré-história, e os dois ajudam a testar a ideia.

Há ainda uma diferença que muda a conta. O tempo entre adormecer e acordar não é o tempo dormindo, porque toda noite tem trechos acordados. A proporção de sono dentro desse período se chama eficiência.

---

### 🧮 O Modelo

São três fontes e quatro contas próprias.

1. **A distância até as oito horas.** Quanto falta para cada grupo sem luz elétrica chegar a oito horas de sono.
2. **As amostras modernas.** Com a média e o desvio das amostras medidas em laboratório, a chance de uma amostra de adultos dormir oito horas ou mais. A conta é P = 1 − Φ((480 − média) / desvio), conferida por simulação (200 mil amostras, semente 42).
3. **A tendência.** Com a inclinação e o valor de p publicados, quanto o sono medido mudou entre 1960 e 2013, e a chance de ter caído 15 minutos ou mais.
4. **Tempo deitado e tempo dormindo.** Quanto sono cabe em oito horas entre adormecer e acordar, com a eficiência medida.

**Fontes:**
- Yetish, G., Kaplan, H., Gurven, M., Wood, B., Pontzer, H., Manger, P. R., Wilson, C., McGregor, R. & Siegel, J. M. (2015). Natural Sleep and Its Seasonal Variations in Three Pre-industrial Societies. *Current Biology*, 25(21), 2862-2868. doi:10.1016/j.cub.2015.09.046. Hadza (Tanzânia), san (Namíbia) e tsimane (Bolívia), com sensor de movimento no pulso por 6 a 28 dias; 45 tsimane e 27 san no texto, 94 pessoas e 1.165 dias no material de divulgação. Lido no manuscrito aceito pelos autores.
- Youngstedt, S. D., Goff, E. E., Reynolds, A. M., Kripke, D. F., Irwin, M. R., Bootzin, R. R., Khan, N. & Jean-Louis, G. (2016). Has adult sleep duration declined over the last 50+ years? *Sleep Medicine Reviews*, 28, 65-81. 168 estudos de 1960 a 2013, 257 amostras, 6.052 adultos saudáveis, sono medido por polissonografia ou sensor de movimento. Lido no artigo completo.
- Watson, N. F. et al. (2015). Recommended Amount of Sleep for a Healthy Adult: A Joint Consensus Statement of the American Academy of Sleep Medicine and Sleep Research Society. *Journal of Clinical Sleep Medicine*, 11(6), 591-592 (também em *Sleep*, 38(6)). Painel de 15 especialistas, 5.314 artigos revisados. Lido no documento completo.

**Nota metodológica sobre o fator ×0,80.** Não se aplica. O sono foi medido por aparelho nas duas fontes, e não declarado pelas pessoas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

SEMENTE = 42
N_SIM = 200_000
rng = np.random.default_rng(SEMENTE)

VERMELHO = '#c0392b'
DOURADO = '#c8a84b'
VERDE = '#1a5f5a'
CINZA = '#7f8c8d'

print("✅ Bibliotecas carregadas")

In [ ]:
# --- DADOS DA LITERATURA ---

# Yetish et al. (2015), Current Biology 25(21), 2862-2868
# Três povos sem luz elétrica (hadza, san, tsimane), sensor de movimento no pulso
n_tsimane = 45
n_san = 27
n_total_divulgado = 94            # total citado no material de divulgação do estudo
dias_divulgados = 1165
sono_min_h, sono_max_h = 5.7, 7.1          # horas de sono por noite, faixa entre grupos e estações
periodo_min_h, periodo_max_h = 6.9, 8.5    # horas entre adormecer e acordar
eficiencia_min, eficiencia_max = 0.81, 0.86
inverno_tsimane_min = 56          # minutos a mais no inverno
inverno_san_min = 53
horas_apos_por_do_sol = 3.3

# Youngstedt et al. (2016), Sleep Medicine Reviews 28, 65-81
# Sono medido por aparelho em adultos saudáveis, estudos de 1960 a 2013
n_estudos = 168
n_pontos = 257
n_pessoas = 6052
media_habitual, dp_habitual = 435.1, 37.4   # minutos, horários habituais (média e DP entre amostras)
media_fixo, dp_fixo = 443.3, 31.7           # minutos, horários fixos de laboratório
inclinacao = 0.03                           # minutos por ano
p_inclinacao = 0.56
ano_inicial, ano_final = 1960, 2013

# Watson et al. (2015), consenso AASM e SRS: recomendação para adultos
recomendacao_h = 7.0
n_especialistas = 15
n_artigos_revisados = 5314

OITO_HORAS = 480
aplica_fator_080 = False

print("=" * 70)
print("  DADOS DA LITERATURA - AS OITO HORAS DE SONO")
print("=" * 70)
print("\n  Yetish et al. (2015), Current Biology. Três povos sem luz elétrica:")
print(f"  → Sono por noite: {sono_min_h:.1f} a {sono_max_h:.1f} horas")
print(f"  → Entre adormecer e acordar: {periodo_min_h:.1f} a {periodo_max_h:.1f} horas")
print(f"  → Eficiência do sono: {eficiencia_min*100:.0f}% a {eficiencia_max*100:.0f}%")
print(f"  → A mais no inverno: {inverno_tsimane_min} min (tsimane) e {inverno_san_min} min (san)")
print(f"  → Adormecem {horas_apos_por_do_sol:.1f} horas depois do pôr do sol")
print("\n  Youngstedt et al. (2016), Sleep Medicine Reviews. Sono medido, 1960 a 2013:")
print(f"  → {n_estudos} estudos, {n_pontos} amostras, " + f"{n_pessoas:,}".replace(',', '.') + " adultos")
print(f"  → Horários habituais: {media_habitual:.1f} min (DP {dp_habitual:.1f})  ·  horários fixos: {media_fixo:.1f} min (DP {dp_fixo:.1f})")
print(f"  → Tendência: {inclinacao:.2f} min por ano (p = {p_inclinacao:.2f})")
print("\n  Watson et al. (2015), consenso de 15 especialistas:")
print(f"  → Recomendação para adultos: {recomendacao_h:.0f} horas ou mais por noite")
print(f"\n  Fator ×0,80 aplicado: {'sim' if aplica_fator_080 else 'não'}")
print("=" * 70)

In [ ]:
# --- O MODELO ---
# Quatro contas próprias sobre os números publicados. Semente 42, 200 mil sorteios.

def hm(minutos):
    h = int(minutos // 60)
    m = int(round(minutos - 60 * h))
    return f"{h} h {m:02d} min"

# ---------- Camada 1: a distância até as oito horas ----------
falta_min = OITO_HORAS - sono_max_h * 60     # o grupo que mais dorme
falta_max = OITO_HORAS - sono_min_h * 60     # o grupo que menos dorme
meio_faixa = (sono_min_h + sono_max_h) / 2 * 60

# ---------- Camada 2: as amostras modernas ----------
# Distribuição das médias das amostras (não de pessoas), suposta normal
p_oito_habitual = 1 - stats.norm.cdf(OITO_HORAS, media_habitual, dp_habitual)
p_sete_habitual = 1 - stats.norm.cdf(recomendacao_h * 60, media_habitual, dp_habitual)
p_oito_fixo = 1 - stats.norm.cdf(OITO_HORAS, media_fixo, dp_fixo)
amostras_sim = rng.normal(media_habitual, dp_habitual, N_SIM)
p_oito_sim = (amostras_sim >= OITO_HORAS).mean()
p_faixa_sem_luz = ((amostras_sim >= sono_min_h * 60) & (amostras_sim <= sono_max_h * 60)).mean()

# ---------- Camada 3: a tendência em mais de 50 anos ----------
# Erro padrão reconstruído da inclinação e do valor de p publicados (aproximação normal)
z_p = stats.norm.ppf(1 - p_inclinacao / 2)
ep_inclinacao = inclinacao / z_p
anos = ano_final - ano_inicial
mudanca = inclinacao * anos
ic_mudanca = ((inclinacao - 1.96 * ep_inclinacao) * anos, (inclinacao + 1.96 * ep_inclinacao) * anos)
incl_sim = rng.normal(inclinacao, ep_inclinacao, N_SIM) * anos
p_queda_15 = (incl_sim <= -15).mean()
p_queda_30 = (incl_sim <= -30).mean()

# ---------- Camada 4: oito horas na cama não são oito horas de sono ----------
sono_em_8h = (OITO_HORAS * eficiencia_min, OITO_HORAS * eficiencia_max)
cama_para_8h = (OITO_HORAS / eficiencia_max, OITO_HORAS / eficiencia_min)

print("=" * 70)
print("  MODELO - RESULTADO")
print("=" * 70)
print("\n  CAMADA 1 · A distância até as oito horas (povos sem luz elétrica)")
print(f"  → O grupo que mais dorme fica a {hm(falta_min)} das oito horas")
print(f"  → O grupo que menos dorme fica a {hm(falta_max)}")
print(f"  → Meio da faixa: {hm(meio_faixa)}")
print(f"  → Grupos e estações que chegam a oito horas de sono: nenhum")
print("\n  CAMADA 2 · As amostras modernas (sono medido, horários habituais)")
print(f"  → Média: {hm(media_habitual)}")
print(f"  → P(uma amostra medir oito horas ou mais): {p_oito_habitual*100:.1f}%  (simulação {p_oito_sim*100:.1f}%)")
print(f"  → P(uma amostra medir sete horas ou mais): {p_sete_habitual*100:.1f}%")
print(f"  → P(uma amostra cair na faixa dos povos sem luz elétrica): {p_faixa_sem_luz*100:.1f}%")
print(f"  → Com horário fixo de laboratório, P(oito horas ou mais): {p_oito_fixo*100:.1f}%")
print(f"\n  CAMADA 3 · A tendência de {ano_inicial} a {ano_final}")
print(f"  → Erro padrão reconstruído: {ep_inclinacao:.3f} min por ano")
print(f"  → Mudança em {anos} anos: {mudanca:+.1f} min, IC 95% [{ic_mudanca[0]:+.1f}; {ic_mudanca[1]:+.1f}]")
print(f"  → P(queda de 15 minutos ou mais): {p_queda_15*100:.2f}%")
print(f"  → P(queda de 30 minutos ou mais): {p_queda_30*100:.2f}%")
print("\n  CAMADA 4 · Tempo deitado e tempo dormindo")
print(f"  → Oito horas entre adormecer e acordar rendem de {hm(sono_em_8h[0])} a {hm(sono_em_8h[1])} de sono")
print(f"  → Para dormir oito horas seriam precisas de {hm(cama_para_8h[0])} a {hm(cama_para_8h[1])}")
print("=" * 70)

In [ ]:
# --- VISUALIZAÇÃO - GRÁFICOS SEPARADOS ---

# ── GRÁFICO 1 · A régua das oito horas ──
fig1, ax1 = plt.subplots(figsize=(12, 8))
linhas = [
    ('Povos sem luz elétrica\nsono por noite', sono_min_h, sono_max_h, VERDE),
    ('Povos sem luz elétrica\nentre adormecer e acordar', periodo_min_h, periodo_max_h, CINZA),
    ('Adultos em estudos de 1960 a 2013\nsono medido (média ± 1 DP)', (media_habitual - dp_habitual) / 60, (media_habitual + dp_habitual) / 60, DOURADO),
]
for i, (rot, a, b, c) in enumerate(linhas[::-1]):
    ax1.barh(i, b - a, left=a, color=c, alpha=0.85, height=0.45)
    ax1.text(a - 0.06, i, f'{a:.1f} h', va='center', ha='right', fontsize=12, fontweight='bold')
    ax1.text(b + 0.06, i, f'{b:.1f} h', va='center', ha='left', fontsize=12, fontweight='bold',
             bbox=dict(facecolor='white', edgecolor='none', pad=2), zorder=5)
ax1.plot([media_habitual / 60] * 2, [-0.22, 0.22], color='#222', lw=2)
ax1.axvline(8, color=VERMELHO, ls='--', lw=2.5)
ax1.text(8.04, 2.42, 'oito horas', color=VERMELHO, fontsize=12, fontweight='bold')
ax1.axvline(recomendacao_h, color='#222', ls=':', lw=1.8)
ax1.text(recomendacao_h - 0.04, 2.42, 'recomendação\nsete horas ou mais', color='#222', fontsize=10, ha='right')
ax1.set_yticks(range(3))
ax1.set_yticklabels([l[0] for l in linhas[::-1]], fontsize=11)
ax1.set_xlim(5, 9.2)
ax1.set_ylim(-0.6, 2.8)
ax1.set_xlabel('Horas por noite', fontsize=12)
ax1.set_title('Nenhum grupo sem luz elétrica chega a oito horas de sono\n'
              'Yetish et al. (2015) e Youngstedt et al. (2016)', fontsize=14, pad=15)
plt.figtext(0.5, 0.01, 'Fontes: Yetish et al., 2015, Current Biology 25(21) · Youngstedt et al., 2016, Sleep Medicine Reviews 28 | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-159-grafico-01-regua.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ── GRÁFICO 2 · As amostras modernas ──
fig2, ax2 = plt.subplots(figsize=(12, 8))
x = np.linspace(300, 570, 1000)
y = stats.norm.pdf(x, media_habitual, dp_habitual)
ax2.plot(x / 60, y, color=VERDE, lw=3)
ax2.fill_between(x / 60, y, where=(x >= OITO_HORAS), color=VERMELHO, alpha=0.45)
ax2.fill_between(x / 60, y, where=(x < OITO_HORAS), color=VERDE, alpha=0.15)
ax2.axvline(8, color=VERMELHO, ls='--', lw=2)
ax2.axvline(media_habitual / 60, color='#222', lw=1.5)
ax2.text(media_habitual / 60 - 0.04, y.max() * 1.02, f'média {hm(media_habitual)}', ha='right', fontsize=12, fontweight='bold')
ax2.text(8.35, y.max() * 0.30, f'{p_oito_habitual*100:.0f}% das amostras\nmedem oito horas\nou mais', color=VERMELHO, fontsize=13, fontweight='bold')
ax2.text(0.03, 0.93, 'P(oito horas ou mais) = 1 − Φ((480 − 435,1) / 37,4)', transform=ax2.transAxes,
         fontsize=12, family='monospace', va='top', bbox=dict(boxstyle='round', facecolor='white', edgecolor=DOURADO))
ax2.set_ylim(0, y.max() * 1.25)
ax2.set_xlabel('Sono medido por noite, em horas (média de cada amostra)', fontsize=12)
ax2.set_ylabel('Densidade', fontsize=12)
ax2.set_title('Em laboratório, uma amostra em cada nove mede oito horas de sono\n'
              '257 amostras de adultos saudáveis, 1960 a 2013', fontsize=14, pad=15)
plt.figtext(0.5, 0.01, 'Modelo próprio sobre a média e o desvio de Youngstedt et al., 2016 | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-159-grafico-02-amostras.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ── GRÁFICO 3 · A tendência ──
fig3, ax3 = plt.subplots(figsize=(12, 8))
t = np.arange(ano_inicial, ano_final + 1)
centro = media_habitual + inclinacao * (t - t.mean())
lo = media_habitual + (inclinacao - 1.96 * ep_inclinacao) * (t - t.mean())
hi = media_habitual + (inclinacao + 1.96 * ep_inclinacao) * (t - t.mean())
ax3.fill_between(t, lo / 60, hi / 60, color=VERDE, alpha=0.2, label='IC 95% da tendência (reconstruído)')
ax3.plot(t, centro / 60, color=VERDE, lw=3, label=f'tendência: {inclinacao:+.2f} min por ano (p = {p_inclinacao:.2f})')
ax3.axhline(8, color=VERMELHO, ls='--', lw=2, label='oito horas')
ax3.axhline(recomendacao_h, color='#222', ls=':', lw=1.8, label='recomendação: sete horas ou mais')
ax3.set_ylim(6.5, 8.4)
ax3.set_xlabel('Ano do estudo', fontsize=12)
ax3.set_ylabel('Sono medido por noite (horas)', fontsize=12)
ax3.set_title('O sono medido não encolheu em mais de 50 anos\n'
              f'Mudança de {mudanca:+.1f} minuto entre {ano_inicial} e {ano_final}, IC 95% [{ic_mudanca[0]:+.1f}; {ic_mudanca[1]:+.1f}]',
              fontsize=14, pad=15)
ax3.legend(loc='lower right', fontsize=11)
plt.figtext(0.5, 0.01, 'Reta ilustrativa a partir da inclinação e do p de Youngstedt et al., 2016 | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-159-grafico-03-tendencia.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

### 💡 O Insight

**Nenhum dos povos sem luz elétrica chegou a oito horas de sono.**

Eles dormem de 5,7 a 7,1 horas por noite. O grupo que mais dorme fica a 54 minutos das oito horas, e o que menos dorme, a mais de duas horas.

Nos laboratórios, a média de 257 amostras de adultos medidas entre 1960 e 2013 é de 7 horas e 15 minutos. Só uma amostra em cada nove chega a oito horas. E o sono medido não encolheu nesse período. A mudança em 53 anos foi de mais 1,6 minuto, com intervalo de menos 3,8 a mais 6,9.

A recomendação oficial também não é oito. O consenso das duas principais sociedades americanas do sono pede sete horas ou mais, e o documento nem cita o número oito.

Onde o oito aparece é no tempo entre adormecer e acordar, que vai de 6,9 a 8,5 horas nesses povos. Oito horas deitado rendem de seis horas e meia a sete horas de sono.

Os dados não dizem quando o número oito surgiu. Dizem que ele não é o padrão de quem vive sem luz elétrica, não é o que se mede em laboratório e não é o que os especialistas recomendam.

A pergunta que fica:

*Você conta as horas que dormiu ou as horas que passou na cama?*

---

### ⚠️ Limitações do Modelo

- Povos que vivem hoje sem luz elétrica não são a pré-história. Eles são a melhor aproximação medida, e vivem perto do Equador, com noites de duração parecida o ano todo.
- O título pergunta por uma origem histórica, e nenhuma das fontes data o surgimento do número oito. O lema operário do século XIX (oito horas de trabalho, oito de lazer e oito de descanso), atribuído a Robert Owen, costuma ser citado como origem. Essa fonte histórica não foi consultada e não entra no modelo.
- O artigo de Yetish publica a faixa entre os grupos (5,7 a 7,1 horas), não a distribuição das pessoas. O modelo não calcula a chance de uma pessoa desses povos dormir oito horas numa noite.
- O total de participantes hadza não aparece no corpo do artigo. Os 94 participantes e os 1.165 dias vêm do material de divulgação.
- O sensor de pulso mede movimento, não sono diretamente, e tende a errar em quem fica parado acordado.
- A média de 435 minutos e o desvio de 37 de Youngstedt descrevem amostras de estudos, não pessoas. "Uma em cada nove" é a parcela de amostras, e supõe distribuição normal.
- A maioria desses estudos mediu o sono em laboratório, o que pode encurtar a noite, e usou voluntários saudáveis, com poucas mulheres. Não é retrato da população.
- O erro padrão da tendência foi reconstruído a partir da inclinação (0,03 minuto por ano) e do valor de p (0,56). É aproximação nossa, e a reta do gráfico é ilustrativa.
- Não ter encontrado queda no sono medido em laboratório não prova que ninguém dorme menos do que antes. Pesquisas com sono declarado, que este dia não usa, mostram resultados variados.
- A recomendação de sete horas ou mais é um consenso de especialistas sobre saúde, não uma medida de quanto as pessoas dormem.
- O historiador Roger Ekirch descreveu um sono em duas partes na Europa anterior à luz elétrica. O estudo de Yetish não encontrou esse padrão. O trabalho de Ekirch não foi lido e não entra no modelo.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*